# E5: landscape shape and search transfer

Held-out error asks whether a model predicts hidden values. This asks whether it
produces a landscape at all.

An additive surrogate fitted on singles has exactly one peak and puts all of its
variance at interaction order one, whatever its held-out error. A benchmark built
on one cannot rank search methods the way the measured landscape does.

Produced by `scripts/run_topology_benchmark.py`. SSMuLA only, since only complete
factorial libraries admit an exact spectrum.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

pd.set_option("display.width", 180)
ROOT = Path.cwd().parents[1] / "notebooks" / "iclr"


In [2]:
topology = pd.read_csv(ROOT / "output_260820_e5" / "topology.csv")
search = pd.read_csv(ROOT / "output_260820_e5" / "search.csv")
ranks = pd.read_csv(ROOT / "output_260820_e5" / "rank_transfer.csv")
print("landscapes", topology.dataset.nunique())

landscapes 16


## Ruggedness

`n_optima` counts genotypes with no fitter one-step neighbour.

In [3]:
topology.groupby("arm")[
    ["n_optima", "peak_fraction", "largest_basin_fraction", "basin_entropy"]
].median().round(4)

,n_optima,peak_fraction,largest_basin_fraction,basin_entropy
arm,,,,
additive,1.0,0.0001,1.0000,0.0000
measured,67.0,0.0072,0.2323,0.7679
posterior,14.0,0.0015,0.2368,0.8484


## Interaction spectrum

`walsh_nonadditive` is the share of variance above order one.

In [4]:
columns = [c for c in topology.columns if c.startswith("walsh_order_")]
topology.groupby("arm")[columns + ["walsh_nonadditive", "walsh_l1_vs_measured"]].median().round(4)

,walsh_order_1,walsh_order_2,walsh_order_3,walsh_order_4,walsh_nonadditive,walsh_l1_vs_measured
arm,,,,,,
additive,1.0000,0.0000,0.0000,0.0000,0.0000,1.6482
measured,0.1759,0.2663,0.4430,0.1549,0.8241,0.0000
posterior,0.2485,0.6070,0.1094,0.0263,0.7515,0.8890


## Per landscape

The measured column is the target every synthetic arm is trying to reproduce.

In [5]:
pivot = topology.pivot_table(index="dataset", columns="arm", values="n_optima", aggfunc="median")
pivot.round(1).to_string()

'arm      additive  measured  posterior\ndataset                               \nDHFR          1.0       3.0       14.5\nGB1           1.0     182.0       43.5\nParD2         1.0       5.0       17.0\nParD3         1.0       4.0       16.5\nT7            1.0      44.0       11.0\nTEV           1.0    1109.0       60.0\nTrpB3A        1.0      60.0       10.5\nTrpB3B        1.0      74.0        3.0\nTrpB3C        1.0      60.0        5.0\nTrpB3D        1.0      43.0       13.5\nTrpB3E        1.0      92.0       16.0\nTrpB3F        1.0      86.0       10.0\nTrpB3G        1.0      76.0       12.0\nTrpB3H        1.0     112.0       10.0\nTrpB3I        1.0      16.0       14.0\nTrpB4         1.0     802.0       46.0'

## Search transfer

Every arm searches the same candidate set under the same budget, so a change in
method ranking can only come from the landscape. Kendall tau compares each
synthetic arm's method ordering against the measured one.

This is where the result turns negative, and it should be reported that way.
Neither synthetic arm preserves the measured ordering, and the additive
surrogate is no worse than the posterior.

In [6]:
ranks.groupby("arm")["kendall_tau"].agg(["median", "mean", "count"]).round(3)

,median,mean,count
arm,,,
additive,0.333,0.208,16
posterior,0.333,0.125,80


In [7]:
ranks.pivot_table(index="dataset", columns="arm", values="kendall_tau").round(3).to_string()

'arm      additive  posterior\ndataset                     \nDHFR        0.333      0.600\nGB1         1.000      0.600\nParD2      -0.333     -0.333\nParD3      -0.333     -0.333\nT7          1.000      0.467\nTEV        -0.333     -0.333\nTrpB3A     -0.333     -0.067\nTrpB3B      0.333      0.600\nTrpB3C      0.333      1.000\nTrpB3D      0.333      0.200\nTrpB3E      0.333     -0.333\nTrpB3F      0.333      0.333\nTrpB3G      0.333     -0.467\nTrpB3H      0.333      0.067\nTrpB3I      0.333     -0.200\nTrpB4      -0.333      0.200'

## Best fitness found, by method and arm

The measured landscape rewards greedy search. Both synthetic arms reward the
ridge-guided search instead, so a method chosen on either surrogate would be the
wrong one. Halving the spectral distance is not enough to move a search
ranking.

In [8]:
search["arm_group"] = search["arm"].str.replace(r"posterior_\d+", "posterior", regex=True)
search.groupby(["arm_group", "method"])["best_fitness"].median().unstack().round(3)

method,greedy,random,ridge_guided
arm_group,,,
additive,0.041,0.031,0.123
measured,0.200,-0.082,-0.050
posterior,0.367,0.359,0.435


## Reading the result

Two findings, pulling in different directions.

Topology fidelity is real. An additive surrogate returns exactly one local
optimum on all sixteen landscapes and carries no variance above interaction
order one. The posterior recovers a median of fourteen optima against sixty-seven
measured, and halves the Walsh distance from 1.65 to 0.89.

Search fidelity is not there yet. Fourteen optima against sixty-seven still
leaves the posterior too smooth for greedy search to show the advantage it holds
on the measured landscape. Ruggedness is the next thing to fix, and the local
optimum count is the metric that says so.

A correctness check worth keeping. The measured cubes give 182 local optima for
GB1 and 802 for TrpB4, matching the frozen reference values in
`docs/project_status.md` exactly. The tensor adapter is reading the same
landscapes the earlier pipeline did.